# Python Ranking Models + Analysis Notebook

This notebook implements the active datathon ranking questions with two approaches:
- weighted scorecards for transparent composites
- predictive models that forecast future composite scores and then rank entities by predicted score

Questions covered here: `2, 3, 4, 9, 10`.


## 1. Data and Modeling Setup

This repo has enough cleaned data to build a first-pass Python ranking pipeline for state and industry rankings. The predictive models use only time-aware splits and forecast future scores rather than ranks directly.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src' / 'ranking').exists() and (candidate / 'models').exists():
            return candidate
    raise RuntimeError('Could not locate repo root containing src/ranking and models.')

ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')

from src.ranking.data import panel_coverage_summary
from src.ranking.pipeline import run_ranking_problem
from models.state_income_momentum import CONFIG as STATE_INCOME_MOMENTUM
from models.industry_wage_upside import CONFIG as INDUSTRY_WAGE_UPSIDE
from models.industry_wage_stability import CONFIG as INDUSTRY_WAGE_STABILITY
from models.industry_household_formation import CONFIG as INDUSTRY_HOUSEHOLD_FORMATION
from models.american_dream_state import CONFIG as AMERICAN_DREAM_STATE
from models.american_dream_industry import CONFIG as AMERICAN_DREAM_INDUSTRY

coverage = panel_coverage_summary()
pd.DataFrame(coverage).T


In [ ]:
PROBLEMS = [
    STATE_INCOME_MOMENTUM,
    INDUSTRY_WAGE_UPSIDE,
    INDUSTRY_WAGE_STABILITY,
    INDUSTRY_HOUSEHOLD_FORMATION,
    AMERICAN_DREAM_STATE,
    AMERICAN_DREAM_INDUSTRY,
]
RESULTS = {(cfg.problem_id, cfg.name, horizon): run_ranking_problem(cfg, horizon=horizon) for cfg in PROBLEMS for horizon in (1, 3)}

performance_rows = []
for (_, name, horizon), result in RESULTS.items():
    performance_rows.append({
        'problem': name,
        'horizon_years': horizon,
        'best_model': result['best_model_name'],
        'predictive_mae': result['predictive_metrics']['mae'],
        'naive_mae': result['naive_metrics']['mae'],
        'predictive_rank_corr': result['predictive_metrics']['rank_corr'],
        'naive_rank_corr': result['naive_metrics']['rank_corr'],
        'top10_overlap_scorecard_vs_predictive': result['top_k_overlap'],
    })
performance = pd.DataFrame(performance_rows).sort_values(['problem', 'horizon_years'])
performance


### Feature Engineering Notes

- State models use per-capita personal income as the core level variable.
- Industry models use wages and salaries per full-time equivalent employee.
- Shared features include CAGR, rolling volatility, drawdown, downside-year counts, and recovery ratios.
- National affordability headwinds come from PCE housing/rent, health, education, and housing/construction price series.
- Predictive targets are future weighted-composite scores at 1-year and 3-year horizons.

## 2. State Rankings

In [ ]:
STATE_KEYS = [
    (STATE_INCOME_MOMENTUM.problem_id, STATE_INCOME_MOMENTUM.name, 1),
    (STATE_INCOME_MOMENTUM.problem_id, STATE_INCOME_MOMENTUM.name, 3),
]
for key in STATE_KEYS:
    result = RESULTS[key]
    display(Markdown(f"### {result['config'].name} ({result['horizon']}-year forecast)"))
    display(result['predictive_ranking'].head(10))
    display(result['predictive_ranking'].tail(10))
    display(pd.DataFrame([{
        'best_model': result['best_model_name'],
        'predictive_mae': result['predictive_metrics']['mae'],
        'naive_mae': result['naive_metrics']['mae'],
        'predictive_rank_corr': result['predictive_metrics']['rank_corr'],
    }]))
    display(result['figure'])


## 3. Industry Rankings

In [ ]:
INDUSTRY_KEYS = [
    (INDUSTRY_WAGE_UPSIDE.problem_id, INDUSTRY_WAGE_UPSIDE.name, 1),
    (INDUSTRY_WAGE_UPSIDE.problem_id, INDUSTRY_WAGE_UPSIDE.name, 3),
    (INDUSTRY_WAGE_STABILITY.problem_id, INDUSTRY_WAGE_STABILITY.name, 1),
    (INDUSTRY_WAGE_STABILITY.problem_id, INDUSTRY_WAGE_STABILITY.name, 3),
    (INDUSTRY_HOUSEHOLD_FORMATION.problem_id, INDUSTRY_HOUSEHOLD_FORMATION.name, 1),
    (INDUSTRY_HOUSEHOLD_FORMATION.problem_id, INDUSTRY_HOUSEHOLD_FORMATION.name, 3),
]
for key in INDUSTRY_KEYS:
    result = RESULTS[key]
    display(Markdown(f"### {result['config'].name} ({result['horizon']}-year forecast)"))
    display(result['predictive_ranking'].head(10))
    display(result['predictive_ranking'].tail(10))
    display(pd.DataFrame([{
        'best_model': result['best_model_name'],
        'predictive_mae': result['predictive_metrics']['mae'],
        'naive_mae': result['naive_metrics']['mae'],
        'predictive_rank_corr': result['predictive_metrics']['rank_corr'],
    }]))
    display(result['figure'])


## 4. American Dream Composite Rankings

In [ ]:
for key in [
    (AMERICAN_DREAM_STATE.problem_id, AMERICAN_DREAM_STATE.name, 1),
    (AMERICAN_DREAM_STATE.problem_id, AMERICAN_DREAM_STATE.name, 3),
    (AMERICAN_DREAM_INDUSTRY.problem_id, AMERICAN_DREAM_INDUSTRY.name, 1),
    (AMERICAN_DREAM_INDUSTRY.problem_id, AMERICAN_DREAM_INDUSTRY.name, 3),
]:
    result = RESULTS[key]
    display(Markdown(f"### {result['config'].name} ({result['horizon']}-year forecast)"))
    display(result['predictive_ranking'].head(10))
    display(result['predictive_ranking'].tail(10))
    display(result['figure'])


## 5. Model Comparison and Limitations

In [ ]:
comparison = performance.copy()
comparison['predictive_beats_naive'] = comparison['predictive_mae'] <= comparison['naive_mae']
comparison


Known limitations in this v1:

- National affordability headwinds affect every state or industry equally within a year, so they shift levels more than cross-sectional spread.
- Industry profit data exists in the repo but does not align cleanly enough across all wage industries to use broadly in this first version.
- State coverage in the cleaned input is limited to the geographies present in `cleaned_income_with_state.csv`.

## 6. Final Recommendations

In [ ]:
def summarize_top_entities(result, top_n=5):
    top = result['predictive_ranking'].head(top_n)['entity'].tolist()
    return ', '.join(top)

findings = []
for key, result in RESULTS.items():
    findings.append({
        'problem': result['config'].name,
        'horizon_years': result['horizon'],
        'top_entities': summarize_top_entities(result),
        'best_model': result['best_model_name'],
        'predictive_mae': result['predictive_metrics']['mae'],
        'notes': result['config'].note,
    })
pd.DataFrame(findings).sort_values(['problem', 'horizon_years'])


Recommended next data improvements:

1. Add state-level rent, home price, and broad cost-of-living series to replace the state affordability proxies.
2. Align industry profit series at the same industry grain as wages before using them as predictive inputs.
3. Expand the notebook with narrative interpretation after reviewing the actual top/bottom entities that matter most for the final datathon story.